# VISUALIZE AUGMENTATION OUTPUT

In [ ]:
from pathlib import Path
import numpy as np
import io
from PIL import Image

from emma import AugmentedDataset, EmmaDataset
from emma.augment import HDF5Source
import hdf5plugin

import matplotlib.pyplot as plt

## > DATASET

In [ ]:
DATASET_ROOT = Path("PATH/TO/EMMa")
AUGMENTATION_PATH = Path("PATH/TO/SAM3-H5-FILE")
SOURCE_NAME = "sam3_masks"

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Please set DATASET_ROOT to a valid EMMa dataset directory. Got: {DATASET_ROOT}"
    )

if not AUGMENTATION_PATH.is_file():
    raise FileNotFoundError(
        f"Please set AUGMENTATION_PATH to a valid augmentation file. Got: {AUGMENTATION_PATH}"
    )

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

In [ ]:
masks = HDF5Source(SOURCE_NAME, AUGMENTATION_PATH, mode="r")

In [ ]:
ads = AugmentedDataset(ds).register(masks)

## > VISUALIZE

In [ ]:
num_samples = 20
random_seed = 0
random_sample = True

In [ ]:
augmented_asins = list(ads.source(SOURCE_NAME).keys())

if random_sample:
    rng = np.random.default_rng(random_seed)
    selected_asins = rng.choice(augmented_asins, size=min(num_samples, len(augmented_asins)), replace=False).tolist()
else:
    selected_asins = augmented_asins[:num_samples]

In [ ]:
def load_first_image(ds, asin):
    rec = ds[asin]
    image_dict = ds.get_images(rec)
    _, first_img_bytes = next(iter(image_dict.items()))
    return Image.open(io.BytesIO(first_img_bytes)).convert("RGB"), rec


def show_mask_overlay(ds, asin):
    image, rec = load_first_image(ds, asin)
    image = np.asarray(image)
    mask = ads.get_augmentation(SOURCE_NAME, asin)

    title = ds.get_title(rec)
    categories = ds.get_category_names(rec)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image)
    axes[0].set_title("Image")

    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Mask ({mask.mean():.1%} foreground)")

    axes[2].imshow(image)
    overlay = np.zeros((*mask.shape, 4))
    overlay[..., 1] = 1.0
    overlay[..., 3] = mask.astype(float) * 0.4
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(f"{asin}\n{title}\n{categories}", fontsize=10)
    plt.tight_layout()
    plt.show()


for asin in selected_asins:
    show_mask_overlay(ds, asin)